# COPD Vision — Dataset Exploration Notes

This notebook explains the key decisions made about data preprocessing and class handling before training.

It is a companion to `copd_simple.ipynb`, which shows the data visually. This notebook explains **why** certain values and techniques were chosen.

---
## 1 — Why These Normalization Values?

In every model notebook you will see this line:

```python
T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
```

These are not arbitrary numbers. They are the **mean and standard deviation of the ImageNet dataset**, measured across millions of natural images, channel by channel (Red, Green, Blue).

### What does Normalize actually do?

For each pixel in each channel, it applies:

```
normalized = (pixel_value - mean) / std
```

This shifts and scales the pixel values so that each channel has a mean of 0 and a standard deviation of 1. The model sees consistent input regardless of how bright or dark the original image was.

### Why use ImageNet values instead of computing our own?

All four models (Simple CNN, ResNet-50, EfficientNet-B2, DenseNet-121) use weights pre-trained on ImageNet. During that pre-training, the network learned to expect inputs normalized with those specific values.

If we used different normalization values, the pre-trained weights would be mismatched to our inputs from the very first forward pass — the model would effectively be starting from scratch even though we loaded pre-trained weights.

By using the same normalization the model was originally trained with, we preserve the meaning of its learned features and fine-tuning converges much faster.

### Does this matter for grayscale X-rays?

Yes. Even though chest X-rays are grayscale, we convert them to 3-channel RGB using `.convert("RGB")` before feeding them to the model. This duplication is intentional — it lets us use ImageNet pre-trained architectures without modifying their input layer.

---
## 2 — Why Do We Use Class Weights?

### The problem: class imbalance

In our dataset, the three classes are not equally represented. `copd_simple.ipynb` shows the exact counts, but the pattern is typical of medical imaging datasets — **Normal** images are far more common than **Emphysema**, and **Other** sits somewhere in between.

If we train without adjusting for this imbalance, the model learns a shortcut: it can achieve high accuracy simply by predicting **Normal** for almost every image. It never truly learns what Emphysema looks like because the training signal from those rare samples is drowned out by the majority class.

### The fix: `compute_class_weight("balanced")`

In every model notebook you will see:

```python
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_labels),
    y=train_labels
)
```

`"balanced"` computes a weight for each class using this formula:

```
weight = total_samples / (number_of_classes × samples_in_this_class)
```

So a class with fewer samples gets a **higher weight**, and a class with many samples gets a **lower weight**. The result is that every class contributes equally to the loss regardless of how many examples it has.

These weights are passed to PyTorch's loss function:

```python
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
```

During training, a wrong prediction on an Emphysema image causes a larger loss penalty than a wrong prediction on a Normal image. This forces the model to pay attention to the rare class.

### Why this matters clinically

In a clinical setting, **missing an Emphysema case is far more dangerous than a false positive**. Class weighting pushes the model toward better recall on the disease class — which is exactly what we want for a decision support tool.

---
## Summary

| Decision | Value | Reason |
|---|---|---|
| Normalize mean | `[0.485, 0.456, 0.406]` | ImageNet channel means — matches pre-trained weight expectations |
| Normalize std | `[0.229, 0.224, 0.225]` | ImageNet channel stds — same reason |
| Class weighting | `compute_class_weight("balanced")` | Dataset is imbalanced; prevents model from ignoring rare disease class |
| Input channels | 3 (RGB via `.convert("RGB")`) | Required by ImageNet pre-trained architectures |

These decisions are consistent across all four model notebooks. Changing any of them would require re-evaluating the training results.